# Self-supervised learning

Need the model to teach itself the structure of architectural space by solving pretext tasks. 

## Pretext tasks

### 1. Masked Attribute Prediction (The "What is this room?" Task)

Graph equivalent of BERT. Hide specific features of a node and ask the model to guess them based on its neighbors. 

- **Task**: Randomly mask the feature vector of a node. The model must reconstruct these values.

- **Architectural Intuition**:

    - If a node is connected to 5 other nodes, has a high "betweenness centrality," and is connected to small rectangular nodes... it is likely a Corridor.
 
    - If the model predicts a high "Aspect Ratio" for that masked node, it has successfully learned the concept of a corridor without ever being told the word "Corridor."
 
- **Implementation**

    - Input: Graph with `x[mask_idx] = 0`.
 
    - Output: Predicted `x[mask_idx]`.
 
    - Loss: MSE (Mean Squared Error) between predicted area/shape and actual area/shape.
 
### 2. Edge Re-connection  (The "Where are the doors?" Task)

This focuses on circulation and flow. A floor plan is defined by how you move through it.

- **Task**: Randomly remove 15-20% of the edges (adjacency) from the input graph. The model outputs a probability matrix for every pair of nodes, predicting whether an edge should exist.

- **Architectural Intuition**:

    - Architecture is not random. You usually don't enter a bathroom directly from a kitchen. You almost always enter a closet from a bedroom.
 
    - By forcing the model to predict edges, it learns adjacency rules and privacy gradients (e.g., Public -> Semi-Private -> Private).
 
- **Implementation**: Standard Link Prediction using `VGAE` or a `dot-product` decoder.

### 3. Geometric Jigsaw (The "Spatial Consistency" Task)

Your current graph topology (A connects to B) discards exact relative positions. This task forces the model to remember the geometry.

- **Task**: Given two connected nodes (Room A and Room B), predict the relative angle and distance between their centroids.

- **Architectural Intuition**:

    - "The Bathroom is usually adjacent to the Bedroom (short distance)."
 
    - "The Corridor runs alongside the rooms (specific angular relationships)."
 
- **Implementation**

    - Calculate ground truth relative vectors (*dx, dy*) for every edge. 
 
    - Model predicts these vectors from the node embeddings of *u* and *v*. 
 
    - Loss: MSE (Mean Squared Error) between predicted area/shape and actual area/shape.
 
### 4.  Instance Discrimination (The "Fingerprint" Task - Contrastive)

This is currently the state-of-the-art method (SimCLR/GraphCL).

- **Task**: Given two connected nodes (Room A and Room B), predict the relative angle and distance between their centroids.

- **Architectural Intuition**:

    - Take a Chunk *G*
 
    - Create Augmentation *G1* (Rotated by 90 degrees + Jittered node features).
 
    - Create Augmentation *G2* (Drop 10% edges).
 
    - The model must map *G1* and *G2* to the same vector, while pushing away the vector for a different chunk *H*. 
 
- **Implementation**

    - A floor plan rotated 90 degrees is the same design.
 
    - A floor plan with one missing wall (due to bad DXF parsing) is the same design.
 
    - This forces the embeddings to be invariant to noise and rotation.
 
---

### Recommendation: Which one to choose?

Since you are building this from scratch, I recommend a **Multi-Task Learning** approach combining **#1 (Masking)** and **#2 (Link Prediction)**.

Why?
1.  **Data Reliability:** Your data comes from DXFs. DXFs often have "leaky" geometry (gaps in walls). If you use *Link Prediction* as a pretext task, your model essentially becomes a **DXF Auto-repair tool**. It learns to predict where connections *should* be, even if the geometry didn't perfectly close.
2.  **Feature Richness:** Masking room attributes forces the model to understand that "Geometry determines Function."

#### Proposed Loss Function
$$ Loss = L_{reconstruction} + \lambda L_{masking} $$

1.  **$L_{reconstruction}$**: Try to rebuild the adjacency matrix (learn topology).
2.  **$L_{masking}$**: Try to guess the area/perimeter of hidden nodes (learn geometry).

This creates a **Geometry-Aware Topological Embedding**—exactly what you need for architectural analysis.